# [LAB] Streamlit RAG 챗봇 실습

보험 약관을 근거로 답하는 RAG 챗봇을 웹 화면으로 띄웁니다. Streamlit 앱은 노트북 셀에서 바로 돌지 않으므로, 셀이 앱 파일을 쓰고 그다음 셀이 그 파일을 실행해 브라우저에 앱을 엽니다.

- **2장 버튼 앱으로 재실행 보기**: 버튼을 누를 때마다 스크립트 전체가 다시 실행되는 모습을 봅니다. `# ✍️ CHANGE_HERE:` 줄의 늘어나는 값을 바꿔 보고, 일반 변수로 세면 어떻게 되는지 AI에게 묻습니다.
- **3장 보험 약관 QA 챗봇**: 대화 기록 저장 두 곳과 답변 스트리밍 한 곳에서 보기를 하나씩 고릅니다. 앱 제목 아래 안내 문구도 바꿔 봅니다.

앱을 띄운 셀은 멈출 때까지 끝나지 않습니다. 다음 셀로 넘어가기 전에 그 셀의 실행을 중지(■)합니다.

In [ ]:
%cd ../../../course/Chapter01/notebooks

**목차**

- 1. 실습 방법
- 2. 버튼 앱으로 재실행 보기
  - 2.1 버튼 앱 쓰고 띄우기
  - 2.2 AI에게 물어보기
- 3. 보험 약관 QA 챗봇
  - 3.1 챗봇 앱 보기 고르고 쓰기
- 4. 과제: 챗봇 완성
- 5. 정리

In [ ]:
from config.check import check_env

# 값은 출력하지 않고, 필요한 설정 이름이 있는지만 봅니다
check_env(["EMBED_PROVIDER"])

**볼 지점**
- "준비 완료"가 나오면 다음으로 넘어갑니다.
- "준비가 덜 되었습니다"가 나오면 ⛔ 줄의 → 안내대로 `.env`를 고친 뒤 바로 위 환경 점검 셀을 다시 실행합니다. 무엇을 고칠지 모르겠으면 그 셀의 출력을 AI에게 붙여 넣고 물어봅니다.
- 키 값은 화면에 나오지 않습니다. 혹시 나오면 셀 출력을 지우고, 그 내용을 다른 곳에 붙여 넣지 않습니다.

LangChain 디버그 모드를 켭니다. 체인과 그래프가 실행될 때 단계마다 입력과 출력이 셀 아래에 출력됩니다.

In [ ]:
from langchain_core.globals import set_debug

# 출력이 너무 길면 True를 False로 바꾸고 이 셀을 다시 실행합니다
set_debug(True)

<details><summary>강사용 메모</summary>

- 원본: `src/raw/Chapter01/06-rag-streamlit/test.py`·`app.py`(raw). 각 파일을 `%%writefile` 한 셀로 넣어 노트북 폴더(`lessons/Chapter01/`)에 `test.py`·`app.py`를 쓰고, 빌더가 만든 `!streamlit run` 셀로 띄웁니다. 순서는 test → app(2026-09-28 사용자 결정, 원본 이름 순과 반대).
- 노트북에서만 원본을 손봤습니다(patch, 원본 파일은 그대로): app.py의 강사 말투 주석 7줄("가우스 모델을 호출할건데…", "한개의 runnable을 받았다", "runnable을 실행하는 방법 -> invoke, stream", "화면에 보여주기 위한 session 관리", "LLM이 이전 대화를 기억한다거나 그런건 X", graph State 비유 등)을 뺐습니다. 솔루션본에도 이 주석은 없습니다. "LLM이 이전 대화를 기억하지 않는다"는 내용은 강의 쪽 session_state 절의 "왜 배우나"로 옮겼습니다.
- app.py의 약관 PDF 경로를 `Path(__file__).parent / 'data' / ...`에서 `Path("data/meritz_pet_insurance.pdf")`로 바꿨습니다. `streamlit run`을 노트북 폴더에서 실행하므로 `lessons/Chapter01/data/meritz_pet_insurance.pdf`(원본과 같은 파일)를 읽습니다.
- 생성되는 `lessons/Chapter01/app.py`·`test.py`는 Git 제외 대상이 아닙니다(`lessons/.gitignore`는 `work/`만). 강사 저장소에서 실행한 뒤에는 커밋에 섞이지 않게 지웁니다.
- `!streamlit`은 셸 PATH에서 찾습니다. 그래서 앱 셀은 `!"{sys.executable}" -m streamlit run …`으로 커널의 파이썬을 씁니다(따옴표는 경로에 공백이 있는 Windows 계정용).
- 앱 셀에 `--server.showEmailPrompt false --server.address 127.0.0.1`을 붙였습니다(2026-09-29). ① 새 PC(`~/.streamlit/credentials.toml` 없음)의 첫 `streamlit run`은 이메일 입력을 기다리며 앱이 뜨지 않습니다(재현됨). Windows의 IPython `!`는 프로세스가 끝나야 출력을 보여 줘 "Email:" 문구도 보이지 않습니다. ② 기본값(모든 인터페이스)이면 Windows 방화벽 허용 창이 뜨고, 같은 망의 다른 PC가 수강생 앱(수강생 키로 모델 호출)에 접속할 수 있습니다. 127.0.0.1이면 출력은 "Local URL/Network URL" 대신 "URL: http://127.0.0.1:8501" 한 줄입니다. `--server.headless true`는 브라우저 자동 열림도 꺼지므로 쓰지 않습니다.
- Windows에서는 ■로 셀을 멈춰도 Streamlit이 8501을 계속 잡고 있어 다음 실행이 8502로 뜰 수 있습니다(포트를 지정하지 않았으므로 Streamlit이 다음 번호로 넘어감). 안내 문구에 반영했습니다. 강의장 Windows PC에서 실제 동작은 수업 전에 확인합니다.
- 성공 기준은 자동 검사하지 않습니다(2026-09-28 결정). CodeMate 지시문 완성본은 주지 않습니다.

</details>

## 1. 실습 방법

코드 셀에서 손댈 곳은 주석으로 표시되어 있습니다.

- `# ✍️ CHANGE_HERE:` 바로 아래 줄의 값을 바꿔 다시 실행해 봅니다.
- `# ✅ CHOOSE_HERE:` 아래에 보기가 주석으로 들어 있습니다. 줄 끝의 번호(1️⃣~4️⃣)와 설명을 보고 하나를 고릅니다. 같은 번호가 붙은 줄을 모두 선택하고 `Ctrl + /`를 눌러 주석을 푼 뒤 실행합니다.

실행한 뒤에는 **성공 기준**과 비교해 봅니다. 🟢는 되어야 하는 것, ⛔는 다시 해야 하는 신호입니다. ⛔이면 푼 줄을 다시 선택하고 `Ctrl + /`로 주석 처리한 뒤 다른 보기를 고릅니다. 마지막으로 접힌 "정답 보기"에서 보기마다의 해설을 읽습니다.

**막혔을 때 AI에게 도움받는 프롬프트 힌트** 오류가 난 셀의 코드와 오류 메시지 전체를 붙여 넣고, 원인과 고칠 곳을 쉬운 말로 알려 달라고 합니다.

## 2. 버튼 앱으로 재실행 보기

버튼 하나와 숫자 하나만 있는 작은 앱입니다. 버튼을 누르면 풍선이 뜨고 누른 횟수가 늘어납니다.

Streamlit은 버튼을 누를 때마다 스크립트를 위에서부터 **전체 다시 실행**합니다. 그래도 횟수가 유지되는 까닭은 `st.session_state`(다시 실행되어도 남는 저장 공간)에 담았기 때문입니다.

**왜 배우나:** 3장 챗봇의 대화 기록도 같은 원리로 남깁니다. 가장 작은 앱에서 재실행을 먼저 눈으로 봅니다.

### 2.1 버튼 앱 쓰고 띄우기

**✍️ CHANGE_HERE: 한 번 누를 때 늘어나는 값을 바꿔 봅니다**

먼저 그대로 실행해 파일을 쓰고, 아래 셀로 앱을 띄워 버튼을 눌러 봅니다. 그다음 앱 셀을 중지(■)합니다. 이때 열려 있던 앱 탭도 멈추니 닫습니다. `# ✍️ CHANGE_HERE:` 줄의 `1`을 다른 수(예: `10`)로 바꿔 두 셀을 다시 실행합니다.

In [ ]:
%%writefile test.py
import streamlit as st

st.title("버튼 누르면 스크립트 전체 다시 실행")

if "clicks" not in st.session_state:
    st.session_state.clicks = 0

if st.button(" 클릭 "):
    # ✍️ CHANGE_HERE: 한 번 누를 때 늘어나는 값을 바꿔 봅니다
    st.session_state.clicks += 1
    st.balloons()

st.write(f"풍선 발사 {st.session_state.clicks}회")

**성공 기준**

- 🟢 셀 아래에 파일을 썼다는 한 줄(Writing … 또는 Overwriting …)이 출력되었다
- 🟢 버튼을 한 번 누를 때마다 바꾼 값만큼 숫자가 늘어난다

위 셀을 실행하면 이 폴더에 앱 파일이 만들어집니다. 아래 셀을 실행하면 잠시 뒤 브라우저에 앱이 저절로 열립니다. 열리지 않으면 브라우저 주소창에 `127.0.0.1:8501`을 넣습니다. Windows에서는 앱이 도는 동안 셀 출력이 비어 있어도 정상입니다. 다시 띄웠을 때 주소 끝이 `8502`인 탭이 열리면 그 탭을 씁니다. 앞 앱이 아직 8501을 쓰고 있다는 뜻입니다. 앱을 멈출 때는 이 셀의 실행을 중지(■)합니다. 멈춰야 다른 셀을 실행할 수 있습니다.

In [ ]:
import sys

# 첫 실행 때 이메일을 묻지 않고, 이 PC에서만 열리게 합니다
!"{sys.executable}" -m streamlit run test.py --server.showEmailPrompt false --server.address 127.0.0.1

**볼 지점**
- 제목 "버튼 누르면 스크립트 전체 다시 실행" 아래에 "클릭" 버튼과 "풍선 발사 0회"가 보입니다.
- 버튼을 누를 때마다 풍선이 뜨고 숫자가 늘어납니다. 버튼을 누를 때마다 스크립트 전체가 다시 실행되지만, 숫자는 `st.session_state`에 있어 사라지지 않습니다.
- 브라우저에서 새로고침하면 숫자가 0으로 돌아갑니다. `st.session_state`는 브라우저 연결(세션)이 이어지는 동안만 남습니다.

<details><summary>강사용 메모</summary>

- 솔루션본 `test.py`는 `clicks = 0` 일반 변수 버전이라(if 줄 주석 처리) 누를 때마다 0에서 다시 시작해 "1회"에 머뭅니다. 아래 "AI에게 물어보기"의 정답 확인용으로 쓸 수 있습니다. 수강생 화면에는 넣지 않았습니다.
- 원본 버튼 글자는 앞뒤에 공백이 있는 `' 클릭 '`입니다.

</details>

### 2.2 AI에게 물어보기

**프롬프트 힌트** 2.1에서 앱 파일을 쓰는 셀의 코드를 붙여 넣고, `st.session_state.clicks` 대신 일반 변수 `clicks = 0`으로 세면 버튼을 여러 번 눌렀을 때 숫자가 어떻게 되는지 물어봅니다. 코드는 고치지 말고 결과와 이유만 설명해 달라고 합니다.

**성공 기준**

- 🟢 "누를 때마다 0에서 다시 시작해, 한 번 누를 때 늘어나는 값(2.1에서 바꾼 수)에 머문다"고 답한다
- 🟢 이유로 "버튼을 누를 때마다 스크립트 전체가 다시 실행되어 `clicks = 0`이 다시 실행된다"를 짚는다
- ⛔ "숫자가 계속 늘어난다"고 답한다 → 버튼을 누르면 스크립트가 어디서부터 다시 실행되는지 다시 묻습니다
- ⛔ 코드를 고쳐 왔다 → 고치지 말고 설명만 해 달라고 다시 적습니다

## 3. 보험 약관 QA 챗봇

메리츠 펫보험 약관(PDF)을 근거로 답하는 챗봇입니다. 코드는 두 부분입니다.

- **RAG 파이프라인 준비** `load_rag_chain()`: 약관 PDF를 읽고 조각으로 나눈 뒤 임베딩해 벡터 저장소에 넣고, 검색 → 프롬프트 → LLM → 글자만 꺼내기로 이어지는 체인을 만듭니다. `@st.cache_resource`가 붙어 있어 한 번 만든 체인을 다시 씁니다.
- **Streamlit 화면**: 제목과 안내 문구, 이전 대화 다시 그리기, 입력창, 답변 말풍선입니다.

질문 저장, 답변 스트리밍, 답변 저장을 한 셀 안에서 모두 고른 뒤 파일을 쓰고 앱을 띄워 확인합니다. 아무것도 고르지 않고 띄우면 오류 없이 앱이 뜨지만, 답변 말풍선이 비어 있고 질문도 기록에 남지 않습니다.

시작하기 전에 2장의 버튼 앱 셀이 아직 돌고 있으면 실행을 중지(■)합니다. 앱 셀이 도는 동안에는 이 장의 셀이 실행되지 않습니다.

### 3.1 챗봇 앱 보기 고르고 쓰기

**✍️ CHANGE_HERE: 제목 아래 안내 문구를 바꿔 봅니다**

이 절은 맨 나중에 합니다. 세 곳에서 보기를 고르고 앱이 동작하는 것을 확인한 다음, 앱 셀을 중지(■)하고 `# ✍️ CHANGE_HERE:` 줄의 따옴표 안 문구를 바꿔(예: "펫보험 약관에 있는 내용만 답합니다.") 파일 쓰기 셀과 앱 셀을 다시 실행합니다.

**✅ 사용자 메시지를 session_state에 저장해 주세요.**

- 🟢 두 번째 질문을 해도 첫 질문 말풍선이 사람 쪽에 남아 있다
- ⛔ 앞 질문이 사라지거나 AI 쪽으로 옮겨 가거나 빨간 오류 상자가 뜬다 → 앱을 멈추고 다른 보기를 고릅니다

**✅ RAG 체인의 stream() 결과를 Streamlit에 출력해 주세요.**

- 🟢 답변이 한 번에 나오지 않고 글자가 이어지며 나타난다
- ⛔ 빨간 오류 상자가 뜨거나 답변 말풍선이 비어 있다 → 앱을 멈추고 다른 보기를 고릅니다

**✅ AI 응답을 session_state에 저장해 주세요.**

- 🟢 다음 질문을 해도 이전 AI 답변 말풍선이 남아 있다
- ⛔ 이전 답변 자리에 내 질문이 보이거나 빨간 오류 상자가 뜬다 → 앱을 멈추고 다른 보기를 고릅니다

In [ ]:
%%writefile app.py
from pathlib import Path

import streamlit as st
from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_community.document_loaders import PDFMinerLoader
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_text_splitters import RecursiveCharacterTextSplitter

from config.embeddings import get_embeddings
from config.llm import get_llm

load_dotenv()


# ── RAG 파이프라인 초기화 (앱 시작 시 한 번만 실행) ─────────────────────────
@st.cache_resource
def load_rag_chain():
    PDF_PATH = Path("data/meritz_pet_insurance.pdf")

    loader = PDFMinerLoader(str(PDF_PATH.resolve()))
    documents = loader.load()

    splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=75)
    chunks = splitter.split_documents(documents)

    embeddings = get_embeddings()
    vectorstore = Chroma.from_documents(chunks, embedding=embeddings)
    retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

    llm = get_llm()
    prompt = ChatPromptTemplate.from_messages(
        [
            (
                "system",
                "당신은 문서 기반 QA 어시스턴트입니다. 주어진 context만 근거로 답하세요.\n\ncontext:\n{context}",
            ),
            ("human", "{question}"),
        ]
    )

    def format_docs(docs):
        return "\n\n".join(doc.page_content for doc in docs)

    chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )
    return chain


# ── Streamlit UI ──────────────────────────────────────────────────────────────

st.title("보험 약관 QA 챗봇")
# ✍️ CHANGE_HERE: 제목 아래 안내 문구를 바꿔 봅니다
st.caption("메리츠 펫보험 약관을 기반으로 답변합니다.")

# 대화 기록을 session_state에 저장 (새로고침 전까지 유지)
if "messages" not in st.session_state:
    st.session_state.messages = []

# 이전 대화 표시
for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.write(msg["content"])

# 사용자 입력
if prompt := st.chat_input("질문을 입력하세요 (예: 보험료는 어떻게 계산되나요?)"):
    # 사용자 메시지 표시
    # ✅ CHOOSE_HERE: 사용자 메시지를 session_state에 저장해 주세요.
    # (같은 번호의 줄을 모두 선택하고 Ctrl + /)
    # st.session_state.messages.append({"role": "assistant", "content": prompt})  # 1️⃣ 질문을 역할 assistant로 추가
    # st.session_state.messages.append({"role": "user", "text": prompt})  # 2️⃣ 질문을 text 키로 추가
    # st.session_state.messages.append({"role": "user", "content": prompt})  # 3️⃣ 질문을 역할 user로 기록 끝에 추가
    # st.session_state.messages = [{"role": "user", "content": prompt}]  # 4️⃣ 기록을 이번 질문 하나로 새로 만듦
    pass  # 그대로 둡니다

    with st.chat_message("user"):
        st.write(prompt)

    # AI 응답 (스트리밍)
    with st.chat_message("assistant"):
        rag_chain = load_rag_chain()
        # ✅ CHOOSE_HERE: RAG 체인의 stream() 결과를 Streamlit에 출력해 주세요.
        # (같은 번호의 줄을 모두 선택하고 Ctrl + /)
        # response = st.write_stream(rag_chain.stream(prompt))  # 1️⃣ stream 조각을 write_stream으로 그림
        # response = st.write(rag_chain.stream(prompt))       # 2️⃣ stream 조각을 write로 그림
        # response = rag_chain.invoke(prompt)                 # 3️⃣ invoke로 답변을 한 번에 받음
        # response = st.write_stream(rag_chain.stream({"question": prompt}))  # 4️⃣ 질문을 question 키 딕셔너리로 넣음

    # ✅ CHOOSE_HERE: AI 응답을 session_state에 저장해 주세요.
    # (같은 번호의 줄을 모두 선택하고 Ctrl + /)
    # st.session_state.messages.append({"role": "assistant", "content": prompt})  # 1️⃣ 질문을 역할 assistant로 추가
    # st.session_state.messages.append(response)              # 2️⃣ 답변 글자만 추가
    # st.session_state.messages.append({"role": "user", "content": response})  # 3️⃣ 답변을 역할 user로 추가
    # st.session_state.messages.append({"role": "assistant", "content": response})  # 4️⃣ 답변을 역할 assistant로 추가

**성공 기준**

- 🟢 셀 아래에 파일을 썼다는 한 줄(Writing … 또는 Overwriting …)이 출력되었다
- 🟢 안내 문구를 바꾼 뒤 다시 띄운 앱에서 제목 아래 글자가 바뀌어 보인다

<details><summary>정답 보기</summary>

**사용자 메시지를 session_state에 저장해 주세요.**

입력창에서 받은 질문(`prompt`)을 대화 기록 `st.session_state.messages`에 넣는 줄입니다. 위쪽 "이전 대화 표시" 코드가 이 기록을 읽어 `msg["role"]` 쪽 말풍선에 `msg["content"]`를 다시 그립니다. 아래 성공 기준은 세 곳을 모두 고른 뒤 앱을 띄워 질문을 세 번 이상 해 보며 한꺼번에 확인합니다.

**정답: 3️⃣**

- 1️⃣ 오류 없이 앱이 돌지만, 다음 질문 때 다시 그려진 앞의 질문이 AI 말풍선에 들어갑니다. `role`이 말풍선의 주인을 정하기 때문입니다.
- 2️⃣ 첫 질문은 잘 되지만, 두 번째 질문 때 이전 대화 표시 코드가 `msg["content"]`를 찾지 못해 브라우저에 `KeyError: 'content'`가 뜹니다.
- 3️⃣ 이전 대화 표시 코드가 읽는 `role`, `content` 키로 기록 끝에 덧붙입니다. 질문을 세 번 해도 모든 질문과 답이 남습니다.
- 4️⃣ 덧붙이지 않고 기록을 새 목록으로 바꿉니다. 오류는 없고 두 번째 질문까지는 멀쩡해 보이지만, 세 번째 질문을 하면 첫 질문과 그 답이 사라지고 바로 앞 대화만 남습니다.

**RAG 체인의 stream() 결과를 Streamlit에 출력해 주세요.**

RAG 체인에 질문을 넣고, 만들어지는 답변을 조각으로 받아 말풍선 안에 차례로 그리는 줄입니다. 다 그린 전체 답변은 변수 `response`에 담아 그 아래 답변 저장에서 씁니다.

**정답: 1️⃣**

- 1️⃣ `stream`이 내놓는 조각을 `st.write_stream`이 말풍선에 차례로 그리고, 다 그린 전체 답변 글자를 돌려줍니다. 그 글자가 `response`에 담깁니다.
- 2️⃣ 답변은 화면에 이어지며 나타나지만, `st.write`는 아무것도 돌려주지 않아 `response`가 `None`입니다. 오류 없이, 다음 질문을 하면 이전 답변 자리에 `None`이 보입니다.
- 3️⃣ 답을 받기만 하고 화면에 그리지 않습니다. 오류 없이 답변 말풍선이 비어 있고, 다음 질문을 해야 기록에서 다시 그려진 앞의 답이 나타납니다.
- 4️⃣ 이 체인은 질문 글자 하나를 받아 검색과 `{question}`에 함께 넘깁니다. 딕셔너리를 넣으면 검색에서 막혀 첫 질문부터 브라우저에 `TypeError: 'dict' object is not an instance of 'str'`가 뜹니다.

**AI 응답을 session_state에 저장해 주세요.**

화면에 다 그린 답변(`response`)을 대화 기록에 넣는 줄입니다. 그래야 다음 질문 때 스크립트가 다시 실행되어도 이 답변 말풍선이 다시 그려집니다.

**정답: 4️⃣**

- 1️⃣ 답변 대신 질문(`prompt`)을 저장합니다. 오류 없이, 다음 질문을 하면 이전 답변 말풍선에 내 질문이 그대로 보입니다.
- 2️⃣ 역할·내용 딕셔너리가 아니라 글자만 넣었습니다. 두 번째 질문 때 이전 대화 표시 코드의 `msg["role"]`에서 `TypeError: string indices must be integers, not 'str'`가 브라우저에 뜹니다.
- 3️⃣ 오류 없이 돌지만, 다음 질문 때 다시 그려진 이전 답변이 사람 쪽 말풍선에 들어갑니다.
- 4️⃣ 질문 저장과 같은 모양으로 답변을 AI 역할로 기록 끝에 덧붙입니다. 다음 질문 때 이 답변이 AI 말풍선으로 다시 그려집니다.

</details>

위 셀을 실행하면 이 폴더에 앱 파일이 만들어집니다. 아래 셀을 실행하면 잠시 뒤 브라우저에 챗봇이 저절로 열립니다. 열리지 않으면 브라우저 주소창에 `127.0.0.1:8501`을 넣습니다. Windows에서는 앱이 도는 동안 셀 출력이 비어 있어도 정상입니다. 다시 띄웠을 때 주소 끝이 `8502`인 탭이 열리면 그 탭을 씁니다. 앞 앱이 아직 8501을 쓰고 있다는 뜻입니다. 앱을 멈출 때는 이 셀의 실행을 중지(■)합니다.

In [ ]:
import sys

# 첫 실행 때 이메일을 묻지 않고, 이 PC에서만 열리게 합니다
!"{sys.executable}" -m streamlit run app.py --server.showEmailPrompt false --server.address 127.0.0.1

**볼 지점**
- 첫 질문은 답이 나오기까지 오래 걸립니다. 이때 약관을 읽고 임베딩해 체인을 만듭니다. 두 번째 질문부터는 만들어 둔 체인을 다시 써서 빨리 시작합니다.
- 답변이 한 번에 나오지 않고 글자가 이어지며 나타납니다.
- 펫보험 약관에 관한 질문을 하고, 답변이 약관 내용을 근거로 하는지 봅니다. 약관에 없는 내용을 물으면 "나와 있지 않다"는 답이 나올 수 있습니다. 주어진 context만 근거로 답하라는 지시 때문입니다. 입력창의 예시 질문(보험료 계산)이 그런 경우입니다. 약관에 답이 있는 질문(예: "자기부담금은 얼마인가요?")도 해 보고 두 답을 비교합니다.
- 질문을 두 번 이상 해 봅니다. 앞의 질문과 답변이 화면에 그대로 남아 있어야 합니다.
- 앱 코드에서 오류가 나면 셀 출력이 아니라 브라우저 화면에 빨간 상자로 보입니다. 그 글을 AI에게 붙여 넣고 원인을 물어봅니다.
- 앱을 띄운 셀의 출력에 경고 문구가 보일 수 있습니다. 앱이 동작하면 넘어갑니다.

<details><summary>강사용 메모</summary>

- `load_rag_chain()`은 원본 구분선 주석("앱 시작 시 한 번만 실행")과 달리 첫 질문 때 처음 실행됩니다(`if prompt := ...` 안에서 호출). 그래서 첫 답변만 느립니다.
- 서버를 새로 띄울 때마다 첫 질문에서 약관 전체를 임베딩합니다(메모리 Chroma, 저장 안 함). 사외 openai 경로는 소액 과금이 됩니다.
- 체인에는 이번 질문(`prompt`)만 들어갑니다. `session_state.messages`는 화면 표시용이라 "그건 얼마야?" 같은 이어지는 질문은 앞 대화를 모르고 답합니다. 수강생이 물으면 강의의 session_state "왜 배우나"로 연결합니다.
- 객관식판에서 아무 보기도 고르지 않으면 질문 자리는 `pass`만 남고 나머지 두 자리는 주석뿐이라 오류 없이 앱이 뜹니다. 2026-10-13 AppTest 확인: 질문 말풍선은 보이고 답변 말풍선은 비며, 기록이 쌓이지 않아 다음 질문 때 앞 질문이 사라집니다. 답변 스트리밍만 고르지 않고 답변 저장을 고르면 `NameError: name 'response' is not defined`가 납니다.
- 앱을 띄운 채 파일만 다시 쓰는 것은 커널이 앱 셀에 묶여 있어 할 수 없습니다. 항상 ■ → 파일 쓰기 → 앱 실행 순서입니다. 이때 서버가 새로 떠서 캐시도 비워집니다.
- 2026-09-28 사외(openai) 실행 확인: 입력창 예시 질문 "보험료는 어떻게 계산되나요?"에는 "보험료 계산 방식이나 산출 기준이 나와 있지 않다"는 취지로 답했습니다(검색된 조각에 계산식이 없음). 시연 질문은 약관에 답이 있는 것으로 미리 골라 둡니다. 사내 모델의 답은 미확인입니다.
- 볼 지점의 예시 "자기부담금은 얼마인가요?"는 2026-10-13 AppTest(사외 openai)에서 통원·입원의료비 1일당 1만 원·2만 원… 식으로 약관 내용을 근거로 답했습니다. 입력창 예시(원본 코드)는 슬라이드와 같게 그대로 두었습니다.

</details>

## 4. 과제: 챗봇 완성

이 장의 학습 목표를 코드로 구현해 봅니다. 과제 파일과 자세한 안내(할 일, 프롬프트 힌트, 실행 방법)는 노트북 밖 폴더에 있습니다.

**학습 목표**

- 일반 질문·약관 질문·기억 질문이 한 화면에서 모두 동작하는 최종 챗봇을 완성할 수 있다

**챗봇 5단계**

1. VS Code 탐색기에서 `course/Chapter01/chatbot/README.md`를 열고 안내대로 AI와 함께 작성합니다.
2. VS Code 메뉴 **터미널 → 새 터미널**을 열고(저장소 맨 위 폴더에서 열립니다) 아래 명령으로 실행합니다.

```
uv run --env-file .env streamlit run course/Chapter01/chatbot/stage5.py
```

3. 다 됐으면 AI에게 **"챗봇 5단계 채점해 줘"**라고 말합니다. 학습 목표마다 🟢 통과 / ⛔ 수정 필요 / ❔ 확인 불가와 다음 할 일이 나옵니다.

## 5. 정리

**이 노트북에서 한 것**
- 셀로 앱 파일을 쓰고, 그 파일을 실행해 브라우저에 Streamlit 앱을 띄웠습니다.
- 버튼을 누르거나 질문을 입력할 때마다 스크립트 전체가 다시 실행되는 것을 보았습니다. 남겨야 하는 값은 `st.session_state`에 담았습니다.
- 질문과 답변을 대화 기록에 저장해, 다시 실행되어도 이전 대화가 화면에 남게 했습니다.
- RAG 체인의 답변을 `st.write_stream`으로 글자가 이어지며 나타나게 했습니다.
- 무거운 체인 준비는 `@st.cache_resource`로 한 번만 하고 다시 썼습니다.

보기를 골라 앱을 띄우고, 결과를 성공 기준(🟢/⛔)과 비교한 뒤 정답 보기의 해설로 확인했습니다.

마치기 전에 앱 셀이 아직 돌고 있으면 실행을 중지(■)하고 앱 브라우저 탭을 닫습니다.